# Etapa 2: Selección de técnica de muestreo para la construcción de muestra inicial

**Materia:** Análisis de grandes volúmenes de datos (Gpo 10)  
**Institución:** Tecnológico de Monterrey, Posgrados  
**Equipo 12:**

- Carlos Eduardo Vega Campos (A01797803)
- Marco Emilio Jimenez Jimenez (A01797948)
- Martha Alicia Villalobos Facundo (A01840063)
- Jonathan Javier Monsalve Giraldo (A01840272)

**Profesores:** Dr. Iván Olmos Pineda, Luis Daniel Mendoza  
**Fecha:** 17 de mayo de 2026  
**Dataset:** NYC TLC Yellow Taxi Trip Records 2024-2025

## Objetivo del notebook

Construir una muestra representativa M de la población de viajes Yellow Taxi NYC mediante muestreo estratificado con calibración histórica. La Etapa 1 del proyecto caracterizó el dataset; esta etapa parte de los datos crudos, aplica limpieza basada en los hallazgos de Etapa 1, valida D contra distribuciones históricas verificables, y extrae M mediante `sampleBy` con piso mínimo por estrato. El notebook está pensado para ejecutarse de forma portable, tanto localmente como en Google Colab; todas las rutas de datos son relativas al notebook (`./data/raw`).

## 1. Configuración del entorno

Iniciamos sesión local de Spark. Configuración mínima: subir `spark.sql.debug.maxToStringFields` para evitar truncamiento de logs en agregaciones grandes.

Notebook portable: las rutas son relativas. Funciona en Ubuntu VM local y en Google Colab si previamente se instala Java.

In [1]:
# Dependencias de Python para el notebook. Idempotente.
!pip install -q pyspark findspark pandas matplotlib


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
# Solo en Google Colab: descomentar para instalar Java (la JVM que ejecuta Spark).
# Localmente con env-pyspark esta línea no es necesaria.
# !apt-get install openjdk-8-jdk-headless -qq > /dev/null

In [3]:
import findspark
findspark.init()

from pyspark.sql import SparkSession, functions as F
from pathlib import Path
import json

spark = SparkSession.builder.master("local[*]").getOrCreate()

# Sube el umbral del log de planes (default 25) para evitar WARN benignos al agregar
# múltiples columnas en una sola llamada.
spark.conf.set("spark.sql.debug.maxToStringFields", 100)

print(f"Spark versión: {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/17 15:19:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark versión: 4.1.1


### 1.2 Descarga reproducible de los datos

Reusamos el patrón de Etapa 1: descarga idempotente desde el CDN público de TLC a `./data/raw/` solo si el archivo no existe. Esto mantiene el notebook autosuficiente y portable. En una máquina donde ya se descargaron los 24 parquets mensuales más el catálogo de zonas (por ejemplo, tras ejecutar el notebook de Etapa 1), todas las descargas devuelven `skip` y la celda termina en segundos.

In [4]:
import subprocess

CDN_BASE = "https://d37ci6vzurychx.cloudfront.net"
DATA_DIR = Path("data/raw")
YEARS = [2024, 2025]
LOOKUP_FILE = "taxi_zone_lookup.csv"


def download_if_missing(download_url, target_path):
    """Descarga `download_url` a `target_path` solo si `target_path` no existe.

    Devuelve un string con el estado: 'skip', 'ok' o 'error: <mensaje>'.
    """
    if target_path.exists():
        return "skip"

    target_path.parent.mkdir(parents=True, exist_ok=True)
    result = subprocess.run(
        ["curl", "-sSL", "-o", str(target_path), download_url],
        capture_output=True,
        timeout=900,
    )

    if result.returncode != 0:
        return f"error: curl exit {result.returncode}"

    return "ok"

In [5]:
# Parquets mensuales de viajes
for year in YEARS:
    for month in range(1, 13):
        filename = f"yellow_tripdata_{year}-{month:02d}.parquet"
        url = f"{CDN_BASE}/trip-data/{filename}"
        target = DATA_DIR / filename
        status = download_if_missing(url, target)
        print(f"{status:>6}  {filename}")

# Tabla de referencia de zonas de taxi
url = f"{CDN_BASE}/misc/{LOOKUP_FILE}"
target = DATA_DIR / LOOKUP_FILE
status = download_if_missing(url, target)
print(f"{status:>6}  {LOOKUP_FILE}")

  skip  yellow_tripdata_2024-01.parquet
  skip  yellow_tripdata_2024-02.parquet
  skip  yellow_tripdata_2024-03.parquet
  skip  yellow_tripdata_2024-04.parquet
  skip  yellow_tripdata_2024-05.parquet
  skip  yellow_tripdata_2024-06.parquet
  skip  yellow_tripdata_2024-07.parquet
  skip  yellow_tripdata_2024-08.parquet
  skip  yellow_tripdata_2024-09.parquet
  skip  yellow_tripdata_2024-10.parquet
  skip  yellow_tripdata_2024-11.parquet
  skip  yellow_tripdata_2024-12.parquet
  skip  yellow_tripdata_2025-01.parquet
  skip  yellow_tripdata_2025-02.parquet
  skip  yellow_tripdata_2025-03.parquet
  skip  yellow_tripdata_2025-04.parquet
  skip  yellow_tripdata_2025-05.parquet
  skip  yellow_tripdata_2025-06.parquet
  skip  yellow_tripdata_2025-07.parquet
  skip  yellow_tripdata_2025-08.parquet
  skip  yellow_tripdata_2025-09.parquet
  skip  yellow_tripdata_2025-10.parquet
  skip  yellow_tripdata_2025-11.parquet
  skip  yellow_tripdata_2025-12.parquet
  skip  taxi_zone_lookup.csv


In [6]:
files = sorted(DATA_DIR.glob("*"))
total_bytes = sum(f.stat().st_size for f in files)

for f in files:
    size_mb = f.stat().st_size / (1024 ** 2)
    print(f"{size_mb:>8.1f} MB   {f.name}")

print()
print(f"Archivos: {len(files)} (esperados: {len(YEARS) * 12 + 1})")
print(f"Tamaño total: {total_bytes / (1024 ** 3):.2f} GB")

     0.0 MB   taxi_zone_lookup.csv
    47.6 MB   yellow_tripdata_2024-01.parquet
    48.0 MB   yellow_tripdata_2024-02.parquet
    57.3 MB   yellow_tripdata_2024-03.parquet
    56.4 MB   yellow_tripdata_2024-04.parquet
    59.7 MB   yellow_tripdata_2024-05.parquet
    57.1 MB   yellow_tripdata_2024-06.parquet
    49.9 MB   yellow_tripdata_2024-07.parquet
    48.7 MB   yellow_tripdata_2024-08.parquet
    58.3 MB   yellow_tripdata_2024-09.parquet
    61.4 MB   yellow_tripdata_2024-10.parquet
    57.8 MB   yellow_tripdata_2024-11.parquet
    58.7 MB   yellow_tripdata_2024-12.parquet
    56.4 MB   yellow_tripdata_2025-01.parquet
    57.5 MB   yellow_tripdata_2025-02.parquet
    66.7 MB   yellow_tripdata_2025-03.parquet
    64.2 MB   yellow_tripdata_2025-04.parquet
    74.2 MB   yellow_tripdata_2025-05.parquet
    70.1 MB   yellow_tripdata_2025-06.parquet
    63.8 MB   yellow_tripdata_2025-07.parquet
    59.4 MB   yellow_tripdata_2025-08.parquet
    69.1 MB   yellow_tripdata_2025-09.parquet

## 2. Carga del dataset y downcast del esquema

Cargamos los 24 parquets mensuales dejando que Spark infiera los tipos nativos, con `mergeSchema=True` para conservar `cbd_congestion_fee` (columna que solo aparece en archivos a partir de 2025-01-05). Inmediatamente después aplicamos el downcast validado en Etapa 1 sección 6 con `selectExpr`: `tinyint` para enums, `smallint` para zonas, `float` para montos. El downcast reduce del orden de 72 bytes por fila respecto a los tipos por defecto (`long` para enums, `double` para montos), lo que equivale a aproximadamente 6 GB de presión de memoria evitable sobre el dataset completo.

**Por qué no pasamos un `StructType` explícito a `.schema()`:** TLC publica algunos parquets con `trip_distance` como Parquet `DOUBLE` y otros como `FLOAT`. Forzar el esquema a `FloatType` en la lectura provoca `PARQUET_COLUMN_DATA_TYPE_MISMATCH` porque Spark no aplica coerción `double → float` al nivel de read. La estrategia robusta es inferir + cast con `selectExpr`, que sí realiza la conversión al nivel de proyección. Es el mismo patrón documentado en Etapa 1.

Referencia oficial sobre `mergeSchema`: https://spark.apache.org/docs/latest/sql-data-sources-parquet.html#schema-merging

In [7]:
parquet_paths = sorted(str(p) for p in DATA_DIR.glob("yellow_tripdata_*.parquet"))

df_native = (spark.read
    .option("mergeSchema", "true")
    .parquet(*parquet_paths))

zones = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(DATA_DIR / "taxi_zone_lookup.csv")))

print(f"Archivos parquet cargados: {len(parquet_paths)}")
print(f"Particiones del DataFrame de viajes: {df_native.rdd.getNumPartitions()}")
print(f"Zonas en el catálogo: {zones.count()}")

Archivos parquet cargados: 24
Particiones del DataFrame de viajes: 22
Zonas en el catálogo: 265


In [8]:
df_raw = df_native.selectExpr(
    "cast(VendorID as tinyint) VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "cast(passenger_count as tinyint) passenger_count",
    "cast(trip_distance as float) trip_distance",
    "cast(RatecodeID as tinyint) RatecodeID",
    "store_and_fwd_flag",
    "cast(PULocationID as smallint) PULocationID",
    "cast(DOLocationID as smallint) DOLocationID",
    "cast(payment_type as tinyint) payment_type",
    "cast(fare_amount as float) fare_amount",
    "cast(extra as float) extra",
    "cast(mta_tax as float) mta_tax",
    "cast(tip_amount as float) tip_amount",
    "cast(tolls_amount as float) tolls_amount",
    "cast(improvement_surcharge as float) improvement_surcharge",
    "cast(total_amount as float) total_amount",
    "cast(congestion_surcharge as float) congestion_surcharge",
    "cast(Airport_fee as float) Airport_fee",
    "cast(cbd_congestion_fee as float) cbd_congestion_fee",
)

print("Esquema con downcast aplicado:")
df_raw.printSchema()

Esquema con downcast aplicado:
root
 |-- VendorID: byte (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: byte (nullable = true)
 |-- trip_distance: float (nullable = true)
 |-- RatecodeID: byte (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: short (nullable = true)
 |-- DOLocationID: short (nullable = true)
 |-- payment_type: byte (nullable = true)
 |-- fare_amount: float (nullable = true)
 |-- extra: float (nullable = true)
 |-- mta_tax: float (nullable = true)
 |-- tip_amount: float (nullable = true)
 |-- tolls_amount: float (nullable = true)
 |-- improvement_surcharge: float (nullable = true)
 |-- total_amount: float (nullable = true)
 |-- congestion_surcharge: float (nullable = true)
 |-- Airport_fee: float (nullable = true)
 |-- cbd_congestion_fee: float (nullable = true)



In [9]:
n_raw = df_raw.count()
print(f"Registros crudos en D: {n_raw:,}")

Registros crudos en D: 89,892,322


El conteo esperado es 89,892,322 registros, consistente con el total observado en Etapa 1 (sección 3) sobre los 24 archivos parquet mensuales. El catálogo de zonas debe traer 265 entradas. Cualquier diferencia indica que algún archivo no se descargó correctamente o que TLC publicó una revisión del dataset.

## 3. Limpieza pre-estratificación

La sección 8 de Etapa 1 consolidó diez problemas de calidad detectados durante el análisis exploratorio sobre D y propuso una corrección para cada uno. En esta etapa ejecutamos esas correcciones para entregar D limpio antes de estratificar y muestrear: los outliers contaminan el cálculo de probabilidades por estrato, y Moorthy (2025) demuestra que sin este paso las correlaciones estructurales colapsan (Pearson r entre distancia y tarifa cercano a cero en datos crudos, frente a valores por encima de 0.8 esperados tras limpieza).

El tratamiento se organiza en tres bloques alineados con la tabla de Etapa 1:

- **Filtros destructivos** (correcciones 1, 2, 4 y 10): eliminan filas con valores físicamente imposibles. Etapa 1 declaró como meta una pérdida combinada inferior al 1.5% del dataset. La pérdida observada se mide empíricamente y la celda de diagnóstico descompone la contribución de cada filtro para que cualquier desviación quede atribuible.
- **Imputaciones** (correcciones 3, 5, 6 y la parte aplicable de 7): rellenan nulos estructurales o valores fuera de dominio sin descartar filas. La diferencia respecto al plan de Etapa 1 está en la corrección 7: Etapa 1 sugirió no imputar las cinco columnas con nulos correlacionados al régimen Flex Fare (era pertinente para EDA); en esta Etapa 2 sí las imputamos a un valor económicamente coherente (cero para montos, código sintético para categóricas) y conservamos en paralelo la bandera `is_flex_fare` (sección 5) para que cualquier análisis posterior pueda aislar el régimen sin haber perdido los registros.
- **Banderas y manejo geográfico** (correcciones 7 y 8): la bandera `is_flex_fare` y el etiquetado de las zonas placeholder 264 y 265 se construyen en la sección 5 como parte de las variables de caracterización, no aquí.

### 3.1 Filtros destructivos

Los umbrales superiores se toman del análisis explícito de Etapa 1 sección 8. Sutileza importante respecto a la versión preliminar de esta celda: Etapa 1 #1 y #2 definieron los rangos válidos como `[0, p99.9]` **incluyendo el cero**, no `(0, p99.9]`. El valor cero corresponde a viajes cancelados o no cobrados que sí son registros válidos (no hubo cobro ni desplazamiento), distintos de los bogus que la corrección #10 ataca de forma específica: `distance = 0` **junto con** `fare > 0` (cobro sin desplazamiento, falla del odómetro). Un filtro `distance > 0` aplicado de forma blanket descartaría todos los viajes cancelados legítimos, sobre-eliminando del orden del 5% al 6% del dataset; conservar `distance = 0` y solo aplicar #10 cuando coexiste con `fare > 0` es la lectura correcta del plan de Etapa 1.

| Etapa 1 # | Columna | Regla | Justificación del umbral |
|---|---|---|---|
| 4 | `tpep_pickup_datetime` | conservar en `[2024-01-01, 2026-01-01)` | El dataset declara cubrir 2024 y 2025; los 59 registros con años 2002, 2007-2009, 2023 y 2026 son errores de reloj del medidor o contaminación cruzada de archivos (Etapa 1 sección 7.4) |
| 1 | `trip_distance` | conservar en `[0, 200]` mi | Cota física razonable: Manhattan a Montauk ida y vuelta son aproximadamente 200 millas (Etapa 1 sección 8). El máximo observado en D es 398,608 mi (16 vueltas a la Tierra), claramente bogus. `distance = 0` se conserva globalmente porque corresponde a viajes cancelados sin cobro |
| 2 | `fare_amount` | conservar en `[0, 1000]` USD | Etapa 1 documentó: la tarifa más alta razonable en NYC ronda los USD 1,000 (viajes interestatales raros). El máximo observado es USD 863,372 y el mínimo USD -2,261 (anulaciones bogus). `fare = 0` se conserva (no-cobros válidos) |
| 2 | `total_amount` | conservar en `[0, 1200]` USD | `total_amount = fare + tip + tolls + recargos`. Con `fare <= 1000`, una propina del 20% y tolls/recargos aeropuerto comunes (~50 USD), el total realista no excede USD 1,200. El máximo observado en D es USD 863,380, hereda los mismos outliers que `fare_amount` |
| 10 | `(trip_distance, fare_amount)` | descartar registros con `distance = 0` **y** `fare > 0` | Etapa 1 sección 9.4 documenta un cluster vertical visible en el scatterplot: alineación densa de puntos en la abscisa cero con tarifas hasta USD 100. Un viaje con tarifa cobrada implica desplazamiento físico; `distance = 0` con `fare > 0` señala falla del odómetro o viaje cancelado mal facturado, no un trayecto real |

Se descarta el filtro previo sobre `passenger_count`: Etapa 1 indicó imputación, no eliminación, para preservar cardinalidad (ver bloque 3.2).

In [10]:
df_filtered = (df_raw
    .filter(F.col("tpep_pickup_datetime") >= F.lit("2024-01-01"))
    .filter(F.col("tpep_pickup_datetime") < F.lit("2026-01-01"))
    .filter(F.col("trip_distance") >= 0)
    .filter(F.col("trip_distance") <= 200)
    .filter(F.col("fare_amount") >= 0)
    .filter(F.col("fare_amount") <= 1000)
    .filter(F.col("total_amount") >= 0)
    .filter(F.col("total_amount") <= 1200)
    .filter(~((F.col("trip_distance") == 0) & (F.col("fare_amount") > 0))))

n_filtered = df_filtered.count()
pct_removed = (n_raw - n_filtered) / n_raw * 100
print(f"Registros tras filtros destructivos: {n_filtered:,}")
print(f"Filas removidas: {n_raw - n_filtered:,} ({pct_removed:.2f}%)")

# La meta aspiracional de Etapa 1 era 1.5%. Si la pérdida observada la excede,
# se documenta en la celda markdown posterior al diagnóstico. El assert solo
# actúa como red de seguridad para casos catastróficos (>15%) que indicarían
# un cambio inesperado en el dataset o un bug en los filtros.
if pct_removed > 1.5:
    print(f"\nAviso: la pérdida ({pct_removed:.2f}%) supera la meta aspiracional de Etapa 1 (1.5%).")
    print("Revisar el diagnóstico por filtro en la celda siguiente para atribuir la causa.")

assert pct_removed < 15.0, f"Filtros removieron {pct_removed:.2f}% > 15%. Algo inesperado pasa con los datos."

Registros tras filtros destructivos: 84,437,138
Filas removidas: 5,455,184 (6.07%)

Aviso: la pérdida (6.07%) supera la meta aspiracional de Etapa 1 (1.5%).
Revisar el diagnóstico por filtro en la celda siguiente para atribuir la causa.


#### Diagnóstico: contribución de cada filtro a la pérdida total

La celda siguiente cuenta, para cada condición de exclusión por separado, cuántas filas elimina sobre `df_raw`. Las cuentas se suman en una sola pasada agregada para no escanear el dataset varias veces. La suma de pérdidas individuales es generalmente mayor que la pérdida combinada `pct_removed` calculada arriba porque varios filtros pueden eliminar las mismas filas (intersección). La utilidad del diagnóstico es identificar cuál filtro es el más agresivo y poder justificar si la pérdida total se aleja de la meta aspiracional del 1.5% que documentó Etapa 1.

In [11]:
filtros_exclusion = [
    ("date < 2024-01-01", F.col("tpep_pickup_datetime") < F.lit("2024-01-01")),
    ("date >= 2026-01-01", F.col("tpep_pickup_datetime") >= F.lit("2026-01-01")),
    ("trip_distance < 0", F.col("trip_distance") < 0),
    ("trip_distance > 200", F.col("trip_distance") > 200),
    ("fare_amount < 0", F.col("fare_amount") < 0),
    ("fare_amount > 1000", F.col("fare_amount") > 1000),
    ("total_amount < 0", F.col("total_amount") < 0),
    ("total_amount > 1200", F.col("total_amount") > 1200),
    ("distance=0 AND fare>0", (F.col("trip_distance") == 0) & (F.col("fare_amount") > 0)),
]

diag = df_raw.agg(*[
    F.sum(cond.cast("int")).alias(name) for name, cond in filtros_exclusion
]).first()

print(f"Total de filas en df_raw: {n_raw:,}\n")
print(f"{'Condición de exclusión':<28} {'Elimina':>12} {'% del total':>14}")
print("-" * 56)
for name, _ in filtros_exclusion:
    n_drop = diag[name] or 0
    pct = n_drop / n_raw * 100
    print(f"{name:<28} {n_drop:>12,} {pct:>13.2f}%")

Total de filas en df_raw: 89,892,322

Condición de exclusión            Elimina    % del total
--------------------------------------------------------
date < 2024-01-01                      57          0.00%
date >= 2026-01-01                      2          0.00%
trip_distance < 0                       0          0.00%
trip_distance > 200                 3,380          0.00%
fare_amount < 0                 3,579,644          3.98%
fare_amount > 1000                     91          0.00%
total_amount < 0                1,583,065          1.76%
total_amount > 1200                    77          0.00%
distance=0 AND fare>0           1,864,632          2.07%


#### Lectura del diagnóstico y decisión de aceptación

La pérdida combinada de 6.07% se descompone en tres causas dominantes, todas documentadas como destructivas en Etapa 1 sección 8:

- **`fare_amount < 0`: 3.98%** (aproximadamente 3.58 millones de registros). Corresponden a anulaciones, créditos y reversiones según Etapa 1 sección 5.2 (mínimo observado USD -2,261). Son transacciones contables, no viajes reales; conservarlas distorsionaría las medias y desviaciones estándar de tarifa y total que validamos en sección 6 contra benchmarks históricos.
- **`distance = 0` con `fare > 0`: 2.07%** (aproximadamente 1.86 millones de registros). El cluster bogus documentado en Etapa 1 sección 9.4 (alineación densa en la abscisa cero del scatterplot). Son fallas del odómetro o viajes cancelados mal facturados, no trayectos físicos.
- **`total_amount < 0`: 1.76%** (aproximadamente 1.58 millones de registros). Solapan en gran medida con `fare_amount < 0` (las reversiones contables tiran ambos campos a valores negativos simultáneamente); la pérdida combinada empírica del 6.07% es menor que la suma lineal del 7.81% precisamente por esa intersección.

Los demás filtros remueven cuotas mínimas: 59 timestamps fuera de rango (Etapa 1 sección 7.4) y outliers extremos por arriba de los topes que también son cifras de docenas a miles, no de millones.

**Decisión: aceptar la pérdida observada del 6.07%.** Tres argumentos sostienen la decisión:

1. **Calidad sobre cantidad.** Etapa 1 declaró los tres bloques dominantes como destructivos en su sección 8. Cumplir la regla de calidad es prioritario sobre minimizar el porcentaje removido. El dataset que sobrevive (84.4 millones de registros) sigue siendo más que suficiente para construir una muestra estratificada robusta de 5 millones en la sección 9.
2. **El sesgo de conservar es peor que la reducción.** Si conserváramos las anulaciones, las medias de `fare_amount` y `total_amount` quedarían sesgadas a la baja y los Pearson r contra `trip_distance` quedarían arrastrados por los valores negativos espurios; la validación histórica contra los USD 13.47 de Etapa 1 (sección 7.1) fallaría por razones contables, no muestrales. Si conserváramos el cluster `distance = 0 ∧ fare > 0`, la celda `manhattan` del estrato quedaría sobrerepresentada por ~1.86M registros falsos, distorsionando el resto del proceso de muestreo.
3. **La meta del 1.5% era aspiracional, no medida.** Etapa 1 declaró la meta sin cuantificar la frecuencia exacta de cada bloque de anomalías sobre el dataset completo. La medición empírica revela que las anulaciones representan aproximadamente el 4% del dataset, no la fracción submarginal que sería compatible con una pérdida combinada del 1.5%. La pérdida observada es un hallazgo cuantitativo de Etapa 2 sobre el dataset 2024-2025 completo, no una falla del plan original de Etapa 1.

`df_filtered` queda con 84,437,138 registros (93.93% del original) y alimenta la sección 3.2 (imputaciones).

### 3.2 Imputaciones

Tras los filtros destructivos, D queda libre de outliers físicamente imposibles pero conserva nulos estructurales y valores fuera de dominio que el muestreo posterior no puede ingerir. Esta sección los corrige sin descartar filas. El resultado es `df_clean`, el DataFrame que alimenta las secciones 4 en adelante.

Las imputaciones eligen valores económicamente coherentes con la semántica del registro, no estadísticas globales arbitrarias: para los nulos del régimen Flex Fare se imputa **cero** en montos (refleja que no hubo cargo cobrado o capturado) y un **código sintético** en categóricas distinguible de los valores oficiales TLC. La bandera `is_flex_fare = (payment_type == 0)` se construye en la sección 5 sobre este `df_clean` y permite a cualquier análisis posterior aislar el régimen sin haber perdido los registros.

| Etapa 1 # | Columna | Regla | Justificación |
|---|---|---|---|
| 3 | `passenger_count` | `NULL` o fuera de `[1, 6]` → `1` | Moda y mediana del dataset son 1 (Etapa 1 sección 7.1); imputar a la moda preserva la cardinalidad sin distorsionar la distribución, a diferencia de la media (~1.4) que no es representable en una variable entera. La capacidad regulatoria TLC es 1-6 pasajeros |
| 5, 6 | `cbd_congestion_fee` | `NULL` o `pickup < 2025-01-05` → `0.0` | El cargo Central Business District entró en vigor el 2025-01-05. Antes de esa fecha la columna no existía en el esquema TLC (46.4% nulos estructurales) o no debía cobrarse (21 registros 2024 con valor no nulo son cobros indebidos). Imputar a cero es el valor económicamente coherente con la regulación vigente en cada fecha |
| 7 | `congestion_surcharge` | `NULL` → `0.0` | Bajo el régimen Flex Fare el TPEP no captura el cargo (17.47% nulos correlacionados con Flex). Cero refleja "no hubo cargo registrado", consistente con el principio del modelo Flex de tarifa upfront sin desglose |
| 7 | `Airport_fee` | `NULL` → `0.0` | Mismo argumento. El cargo aeropuerto bajo Flex queda absorbido en la tarifa fija negociada upfront; cero es económicamente correcto para la columna `Airport_fee` específicamente |
| 7 | `RatecodeID` | `NULL` → `99` | Los códigos TLC oficiales son 1-6 (Standard, JFK, Newark, Nassau/Westchester, Negotiated, Group ride). Flex Fare no aplica RatecodeID por diseño: la tarifa se acuerda upfront sin esquema tarifario. Se asigna 99 como código sintético "no aplica RatecodeID", convención usada por TLC para "unknown" en otros datasets |
| 7 | `store_and_fwd_flag` | `NULL` → `"F"` | Los valores oficiales son `'Y'` (viaje almacenado y reenviado al servidor por falta de conexión del medidor) y `'N'` (transmisión en tiempo real). Bajo Flex el medidor no participa en la transacción, por lo que el flag no aplica. Se asigna `"F"` como código sintético de Flex, distinguible de los valores oficiales |

Tras la imputación todas las columnas listadas quedan sin nulos; la verificación posterior lo confirma. Las restantes columnas del esquema (`VendorID`, montos cobrados, timestamps) no tienen nulos estructurales documentados en Etapa 1.

In [12]:
df_clean = (df_filtered
    .withColumn(
        "passenger_count",
        F.when(F.col("passenger_count").between(1, 6), F.col("passenger_count"))
         .otherwise(F.lit(1).cast("byte"))
    )
    .withColumn(
        "cbd_congestion_fee",
        F.when(
            F.col("cbd_congestion_fee").isNull() | (F.col("tpep_pickup_datetime") < F.lit("2025-01-05")),
            F.lit(0.0).cast("float")
        ).otherwise(F.col("cbd_congestion_fee"))
    )
    .withColumn("congestion_surcharge",
                F.coalesce(F.col("congestion_surcharge"), F.lit(0.0).cast("float")))
    .withColumn("Airport_fee",
                F.coalesce(F.col("Airport_fee"), F.lit(0.0).cast("float")))
    .withColumn("RatecodeID",
                F.coalesce(F.col("RatecodeID"), F.lit(99).cast("byte")))
    .withColumn("store_and_fwd_flag",
                F.coalesce(F.col("store_and_fwd_flag"), F.lit("F")))
)

print("Esquema tras imputaciones (tipos preservados):")
df_clean.printSchema()

Esquema tras imputaciones (tipos preservados):
root
 |-- VendorID: byte (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: byte (nullable = true)
 |-- trip_distance: float (nullable = true)
 |-- RatecodeID: byte (nullable = false)
 |-- store_and_fwd_flag: string (nullable = false)
 |-- PULocationID: short (nullable = true)
 |-- DOLocationID: short (nullable = true)
 |-- payment_type: byte (nullable = true)
 |-- fare_amount: float (nullable = true)
 |-- extra: float (nullable = true)
 |-- mta_tax: float (nullable = true)
 |-- tip_amount: float (nullable = true)
 |-- tolls_amount: float (nullable = true)
 |-- improvement_surcharge: float (nullable = true)
 |-- total_amount: float (nullable = true)
 |-- congestion_surcharge: float (nullable = false)
 |-- Airport_fee: float (nullable = false)
 |-- cbd_congestion_fee: float (nullable = true)



In [13]:
imputed_cols = [
    "passenger_count",
    "cbd_congestion_fee",
    "congestion_surcharge",
    "Airport_fee",
    "RatecodeID",
    "store_and_fwd_flag",
]

null_row = df_clean.agg(*[
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in imputed_cols
]).first()

print("Nulos remanentes por columna tras imputación (esperado: 0 en todas):\n")
print(f"{'Columna':<22} {'Nulos':>10} {'Estado':>8}")
print("-" * 42)
for c in imputed_cols:
    n = null_row[c] or 0
    estado = "OK" if n == 0 else "FAIL"
    print(f"{c:<22} {n:>10,} {estado:>8}")

assert all((null_row[c] or 0) == 0 for c in imputed_cols), \
    "Quedan nulos en columnas imputadas; revisar lógica de imputación."

print("\nImputación completada exitosamente. Todas las columnas objetivo quedan sin nulos.")

Nulos remanentes por columna tras imputación (esperado: 0 en todas):

Columna                     Nulos   Estado
------------------------------------------
passenger_count                 0       OK
cbd_congestion_fee              0       OK
congestion_surcharge            0       OK
Airport_fee                     0       OK
RatecodeID                      0       OK
store_and_fwd_flag              0       OK

Imputación completada exitosamente. Todas las columnas objetivo quedan sin nulos.


## 4. Mapeo de zonas a macro-categorías sin join

La variable de estratificación geográfica es `pu_macro_zone` (sección 5), una categórica de 4 valores (`airport`, `manhattan`, `outer_borough`, `unknown`) derivada del catálogo `taxi_zone_lookup`. La forma directa de derivarla sería un broadcast join del catálogo contra `df_clean`, que agregaría tres columnas string (`pu_borough`, `pu_zone`, `pu_service_zone`) sobre los 84.4 millones de filas, equivalente a un costo estimado de 3-4 GB de presión de memoria sobre `df_clean` por información que solo se usa para construir una categórica de 4 valores.

La alternativa que aplicamos es **resolver el mapeo en el driver con un collect liviano** del catálogo (265 filas, instantáneo) y conservar listas de Python con los `LocationID` por macro categoría. En la sección 5 esas listas se aplican como predicados `.isin(...)` directamente sobre la columna numérica `PULocationID`, sin agregar columnas string al DataFrame principal.

| Aspecto | Broadcast join | Predicado `.isin(lista)` |
|---|---|---|
| Costo por fila | Hash join, equivalente | Hash lookup, equivalente |
| Columnas agregadas a `df_clean` | 3 strings (~30-50 B por fila × 84M ≈ 3-4 GB) | 0 |
| State en el driver | Broadcast de 12 KB | 3 listas Python (~1 KB) |
| Estados posteriores | `df_clean` con columnas redundantes | `df_clean` con IDs numéricos limpios |

El catálogo `zones` queda disponible en sesión por si alguna etapa posterior (Etapa 3 modelado, sección 11 ejemplos por regla) necesita nombres legibles; en ese caso se hace un lookup local sin tocar el DataFrame principal.

### 4.1 Reglas de asignación con precedencia

Las categorías del catálogo se solapan: una misma zona puede satisfacer dos criterios a la vez. **Ejemplo concreto:** JFK Airport tiene `Borough = "Queens"` y `service_zone = "Airports"` en `taxi_zone_lookup`. Si aplicáramos las reglas en orden ingenuo (primero borough, después service_zone), JFK quedaría en `outer_borough` porque Queens es un borough exterior. Pero conceptualmente JFK es aeropuerto: los pickups en JFK siguen el tarifario plano (`RatecodeID = 2`), llevan `Airport_fee > 0`, y son comportamentalmente distintos del resto de Queens. El estrato `airport` debe capturarlos, no `outer_borough`. La misma situación aplica a LaGuardia (Queens / Airports) y Newark (zona dedicada EWR fuera de los cinco boroughs de NYC).

La forma estándar de resolver el solapamiento en PySpark es la cadena `F.when(...).when(...).otherwise(...)` con short-circuit: el primer `when` que matchea gana. Aquí lo replicamos en el driver vía **resta de sets**, equivalente lógica pero sin tocar el DataFrame.

| Precedencia | Macro categoría | Criterio sobre el catálogo | Lógica de exclusión |
|---|---|---|---|
| 1 (más alta) | `airport` | `service_zone IN ('Airports', 'EWR')` | Se calcula primero, sin exclusiones. Capta JFK, LGA y Newark (3 IDs en el dataset 2024-2025) |
| 2 | `unknown` | `LocationID IN (264, 265)` | Placeholders TLC para "Unknown" y "N/A". Se fija por convención sin consultar el catálogo |
| 3 | `manhattan` | `Borough = 'Manhattan'` y `LocationID` **no está** en `airport_ids` ni `unknown_ids` | En la práctica la resta no remueve nada (Manhattan no contiene aeropuertos ni placeholders), pero la expresión queda explícita por simetría y como salvaguarda si TLC agregara una zona aeroportuaria en Manhattan |
| 4 (más baja) | `outer_borough` | `Borough IN ('Brooklyn', 'Queens', 'Bronx', 'Staten Island')` y `LocationID` **no está** en `airport_ids` ni `unknown_ids` | Aquí sí actúa la resta: JFK (132) y LGA (138), ambos en Borough = Queens, salen de este set porque ya fueron asignados a `airport_ids` |

La frase "y no clasificado antes" en la tabla se traduce literalmente a las operaciones de set `set_inferior - airport_ids - unknown_ids` en la celda siguiente. El assert al final verifica que los cuatro sets resultantes no se solapen, es decir, que cada `LocationID` quede asignado a una y solo una macro categoría.

In [14]:
# Collect liviano del catálogo (265 filas) y resolución de la precedencia en el driver.
airport_ids = {
    r.LocationID
    for r in zones.filter(F.col("service_zone").isin("Airports", "EWR")).collect()
}
unknown_ids = {264, 265}

manhattan_ids = {
    r.LocationID
    for r in zones.filter(F.col("Borough") == "Manhattan").collect()
} - airport_ids - unknown_ids

outer_ids = {
    r.LocationID
    for r in zones.filter(F.col("Borough").isin("Brooklyn", "Queens", "Bronx", "Staten Island")).collect()
} - airport_ids - unknown_ids

# Verificación: las cuatro listas particionan el rango 1-265 sin solapamiento y sin huecos.
todos_ids = airport_ids | unknown_ids | manhattan_ids | outer_ids
faltantes = set(range(1, 266)) - todos_ids
solapamiento_total = (
    len(airport_ids) + len(unknown_ids) + len(manhattan_ids) + len(outer_ids)
    - len(todos_ids)
)

print(f"airport       : {len(airport_ids):>3} zonas → {sorted(airport_ids)}")
print(f"unknown       : {len(unknown_ids):>3} zonas → {sorted(unknown_ids)}")
print(f"manhattan     : {len(manhattan_ids):>3} zonas")
print(f"outer_borough : {len(outer_ids):>3} zonas")
print(f"\nCobertura: {len(todos_ids)} de 265 IDs clasificados")
print(f"IDs sin clasificar (caerán a 'unknown' por defecto): {sorted(faltantes) if faltantes else 'ninguno'}")
print(f"Solapamiento entre categorías: {solapamiento_total} (debe ser 0)")

assert solapamiento_total == 0, "Las categorías macro tienen solapamiento; revisar precedencia."

airport       :   3 zonas → [1, 132, 138]
unknown       :   2 zonas → [264, 265]
manhattan     :  69 zonas
outer_borough : 191 zonas

Cobertura: 265 de 265 IDs clasificados
IDs sin clasificar (caerán a 'unknown' por defecto): ninguno
Solapamiento entre categorías: 0 (debe ser 0)


Cuatro `set` de Python (`airport_ids`, `unknown_ids`, `manhattan_ids`, `outer_ids`) particionan los 265 `LocationID` del catálogo sin solapamiento. El assert garantiza la mutua exclusión; los IDs no asignados (si los hubiera) se canalizarían a `unknown` por el `.otherwise(...)` de la sección 5. Estas cuatro listas son los únicos artefactos que sobreviven de esta sección hacia la siguiente: `df_clean` queda intacto sin columnas agregadas.

## 5. Variables de caracterización poblacional

Construimos las cuatro variables del estrato principal y dos banderas auxiliares de validación con base en la evidencia histórica documentada en Etapa 1 y en literatura externa:

- **`pu_macro_zone`** (categórica, 4 valores): colapsa las 265 zonas TLC en macro categorías usando los sets de `LocationID` derivados en la sección 4. TLC HAIL Market Analysis 2013 documenta que aproximadamente 95% de pickups Yellow Taxi ocurren en Manhattan Core + aeropuertos y 5% en outer boroughs. Mantener 265 zonas individuales generaría estratos con celdas casi vacías.

- **`payment_group`** (categórica, 4 valores): separa Flex Fare (`payment_type = 0`, introducido septiembre 2024, aproximadamente 17% del dataset, taxímetro inactivo según TLC Flex Fare Pilot Evaluation 2023) de los pagos metered. Crédito (`payment_type = 1`) y efectivo (`payment_type = 2`) se separan por censura de propinas: TPEP no captura tip en efectivo (Donkor 2020 Stanford). El resto cae en `other`.

- **`day_hour_bucket`** (temporal, 5 valores): captura los picos documentados por Safikhani et al. 2020: 7-9 am (commuting matutino), 16-20 h (pico vespertino, máximo absoluto), 0-5 h (valle nocturno), fines de semana (patrón distinto al laboral), y resto. La cadena de `when` resuelve el solapamiento (un weekday a las 8 am podría matchear `late_night` o `weekday_am`; gana el primero que matchee).

- **`trip_distance_bin`** (continua discretizada, 3 valores): umbrales 1.12 y 12.43 millas según Riascos y Mateos 2020 (conversión de 1.8 km y 20 km usando 1 mi = 1.609 km). Distribución histórica esperada: 41.33% short / 57.49% medium / 1.18% long.

### 5.1 Tabla consolidada de variables de caracterización

| Nombre | Dominio | Estadística conocida (de la población P, de fuentes históricas) | Comentarios |
|---|---|---|---|
| `pu_macro_zone` | 4 valores: `airport`, `manhattan`, `outer_borough`, `unknown` | Manhattan + airport concentran aproximadamente 95% de los pickups Yellow Taxi; outer borough aproximadamente 5%; airport solo aproximadamente 8% del total. Fuente: TLC HAIL Market Analysis 2013 | Colapsa 265 zonas TLC en cuatro macro categorías con precedencia `airport > unknown > manhattan > outer` (ver sección 4.1). La concentración geográfica es estructural por regulación TLC: la flota medallón está físicamente acreditada para operar en Manhattan y aeropuertos. El valor `unknown` corresponde a `LocationID` 264 y 265, placeholders TLC para zonas no clasificables |
| `payment_group` | 4 valores: `credit`, `cash`, `flex`, `other` | Credit share entre metered: 75-78% en período 2017-2019 (Donkor 2020 Stanford). Flex Fare share global: aproximadamente 17% del dataset 2024 post-introducción de septiembre 2024 (TLC Flex Fare Pilot Evaluation 2023). Cash share decreciente por adopción de pago electrónico post-COVID | `payment_type = 0` corresponde a Flex Fare (taxímetro inactivo, tarifa upfront). `payment_type = 1` es crédito, 2 es efectivo, 3-6 caen en `other`. La separación cash vs credit es importante por la censura de propinas: TPEP no captura tip en efectivo |
| `day_hour_bucket` | 5 valores: `weekday_am`, `weekday_pm_peak`, `late_night`, `weekend`, `other` | Picos de demanda documentados: 7-9 am (commuting matutino), 16-20 h (pico vespertino, máximo absoluto sobre las 24 horas), 0-5 h (valle nocturno). Patrones de fin de semana cualitativamente distintos al laboral. Fuente: Safikhani, Kamga, Mudigonda, Faghih y Moghimi 2020 (International Journal of Forecasting) | La cadena de `when` con short-circuit resuelve solapamientos (un weekday a las 8 am matchea primero `late_night` solo si está en 0-5h; aquí no aplica, gana `weekday_am`). El bucket `other` recoge weekdays 10-16h y 20-24h (horario laboral no-pico) |
| `trip_distance_bin` | 3 valores: `short`, `medium`, `long` | Distribución histórica de viajes: 41.33% short, 57.49% medium, 1.18% long. Fuente: Riascos y Mateos 2020 sobre datos 2010-2013 (Scientific Reports) | Umbrales 1.12 mi y 12.43 mi (conversión de 1.8 km y 20 km de Riascos y Mateos, usando 1 mi = 1.609 km). La distribución empírica observada en D 2024-2025 difiere significativamente del benchmark histórico por displacement FHV de viajes cortos intra-Manhattan y por la diferencia entre distancia geográfica zona-a-zona (R&M) y distancia odómetro (TLC); ver análisis en la sección 6 (validación histórica de D contra benchmarks poblacionales de P) |

Banderas adicionales (no participan del estrato, sirven para validación posterior y para etapas 3 y 4):
- **`is_flex_fare = (payment_type == 0)`**: identifica el régimen Flex para análisis específicos sin tener que recurrir al valor numérico.
- **`cbd_period_flag`**: separa pre-2025-01-05 de post (entrada en vigor del cargo Central Business District), útil para analizar el efecto regulatorio sobre tarifas en la sección 6 (validación histórica de D contra benchmarks poblacionales de P).

El **estrato compuesto** `stratum_id` es la concatenación `pu_macro_zone | payment_group | day_hour_bucket | trip_distance_bin`, con cardinalidad teórica 4 × 4 × 5 × 3 = 240. Esperamos entre 180 y 220 celdas válidas tras descartar combinaciones imposibles (por ejemplo Flex × airport no existe porque Port Authority prohíbe e-hail en aeropuertos según TLC Flex Fare Pilot Evaluation 2023). La sección 7 (cálculo de probabilidades empíricas por estrato) cuantifica la cardinalidad exacta y enumera explícitamente las combinaciones vacías como probabilidad cero.

In [15]:
# Conversión de los sets de zonas (sección 4) a listas para usar con F.col(...).isin(...).
airport_list = sorted(airport_ids)
unknown_list = sorted(unknown_ids)
manhattan_list = sorted(manhattan_ids)
outer_list = sorted(outer_ids)

df_feat = (df_clean
    .withColumn(
        "pu_macro_zone",
        F.when(F.col("PULocationID").isin(airport_list), "airport")
         .when(F.col("PULocationID").isin(unknown_list), "unknown")
         .when(F.col("PULocationID").isin(manhattan_list), "manhattan")
         .when(F.col("PULocationID").isin(outer_list), "outer_borough")
         .otherwise("unknown")
    )
    .withColumn(
        "payment_group",
        F.when(F.col("payment_type") == 0, "flex")
         .when(F.col("payment_type") == 1, "credit")
         .when(F.col("payment_type") == 2, "cash")
         .otherwise("other")
    )
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("dow", F.dayofweek("tpep_pickup_datetime"))  # 1 = domingo, 7 = sábado
    .withColumn(
        "day_hour_bucket",
        F.when(F.col("pickup_hour").between(0, 5), "late_night")
         .when(F.col("dow").isin(1, 7), "weekend")
         .when(F.col("dow").between(2, 6) & F.col("pickup_hour").between(6, 10), "weekday_am")
         .when(F.col("dow").between(2, 6) & F.col("pickup_hour").between(16, 20), "weekday_pm_peak")
         .otherwise("other")
    )
    .withColumn(
        "trip_distance_bin",
        F.when(F.col("trip_distance") < 1.12, "short")
         .when(F.col("trip_distance") < 12.43, "medium")
         .otherwise("long")
    )
    .withColumn("is_flex_fare", F.col("payment_type") == 0)
    .withColumn(
        "cbd_period_flag",
        F.when(F.col("tpep_pickup_datetime") < F.lit("2025-01-05"), "pre_cbd")
         .otherwise("post_cbd")
    )
    .withColumn(
        "stratum_id",
        F.concat_ws(
            "|",
            F.col("pu_macro_zone"),
            F.col("payment_group"),
            F.col("day_hour_bucket"),
            F.col("trip_distance_bin"),
        )
    )
)

print("Columnas agregadas en df_feat:")
print(sorted(set(df_feat.columns) - set(df_clean.columns)))

df_feat.select(
    "pu_macro_zone", "payment_group", "day_hour_bucket",
    "trip_distance_bin", "stratum_id"
).show(10, truncate=False)

Columnas agregadas en df_feat:
['cbd_period_flag', 'day_hour_bucket', 'dow', 'is_flex_fare', 'payment_group', 'pickup_hour', 'pu_macro_zone', 'stratum_id', 'trip_distance_bin']
+-------------+-------------+---------------+-----------------+----------------------------------+
|pu_macro_zone|payment_group|day_hour_bucket|trip_distance_bin|stratum_id                        |
+-------------+-------------+---------------+-----------------+----------------------------------+
|manhattan    |credit       |late_night     |medium           |manhattan|credit|late_night|medium|
|manhattan    |credit       |late_night     |short            |manhattan|credit|late_night|short |
|manhattan    |cash         |late_night     |medium           |manhattan|cash|late_night|medium  |
|airport      |credit       |late_night     |medium           |airport|credit|late_night|medium  |
|manhattan    |credit       |late_night     |medium           |manhattan|credit|late_night|medium|
|airport      |credit       |la

In [16]:
# Verificación: cada variable categórica del estrato debe tomar exactamente los valores esperados.
# Si aparece un valor inesperado (por ejemplo "unknown" abultado en pu_macro_zone), indica
# un bug en la lógica de when/otherwise o IDs fuera del catálogo.

valores_esperados = {
    "pu_macro_zone": {"airport", "manhattan", "outer_borough", "unknown"},
    "payment_group": {"flex", "credit", "cash", "other"},
    "day_hour_bucket": {"late_night", "weekend", "weekday_am", "weekday_pm_peak", "other"},
    "trip_distance_bin": {"short", "medium", "long"},
}

for col_name, esperados in valores_esperados.items():
    print(f"\n{col_name} (esperados: {sorted(esperados)}):")
    df_feat.groupBy(col_name).count().orderBy(F.desc("count")).show(truncate=False)

    observados = {
        r[col_name] for r in df_feat.select(col_name).distinct().collect()
    }
    extras = observados - esperados
    faltantes = esperados - observados
    assert not extras, f"{col_name} tiene valores no esperados: {extras}"
    if faltantes:
        print(f"  Aviso: faltan valores esperados {faltantes} (puede ser válido si el dataset no los contiene)")


pu_macro_zone (esperados: ['airport', 'manhattan', 'outer_borough', 'unknown']):


+-------------+--------+
|pu_macro_zone|count   |
+-------------+--------+
|manhattan    |73881383|
|airport      |6270530 |
|outer_borough|4077272 |
|unknown      |207953  |
+-------------+--------+




payment_group (esperados: ['cash', 'credit', 'flex', 'other']):


+-------------+--------+
|payment_group|count   |
+-------------+--------+
|credit       |60983386|
|flex         |12546605|
|cash         |9656135 |
|other        |1251012 |
+-------------+--------+




day_hour_bucket (esperados: ['late_night', 'other', 'weekday_am', 'weekday_pm_peak', 'weekend']):


+---------------+--------+
|day_hour_bucket|count   |
+---------------+--------+
|other          |26397022|
|weekend        |19849620|
|weekday_pm_peak|19544098|
|weekday_am     |11501716|
|late_night     |7144682 |
+---------------+--------+




trip_distance_bin (esperados: ['long', 'medium', 'short']):


+-----------------+--------+
|trip_distance_bin|count   |
+-----------------+--------+
|medium           |57104939|
|short            |22641738|
|long             |4690461 |
+-----------------+--------+



La sección 7 (cálculo de probabilidades empíricas por estrato) cuantifica la cardinalidad exacta del estrato (cuántas combinaciones realmente aparecen en `df_feat` de las 240 teóricas) y calcula las probabilidades empíricas `p_D` por combinación. Aquí solo verificamos que las cuatro variables del estrato tomen exclusivamente los valores definidos en la lógica de `when/otherwise`; cualquier valor inesperado señalaría un bug y abortaría la celda anterior.

Las dos banderas auxiliares `is_flex_fare` y `cbd_period_flag` no participan del `stratum_id` pero quedan disponibles en `df_feat` para los análisis de la sección 6 (validación histórica de D contra benchmarks poblacionales de P) y de aislamiento del régimen Flex Fare en etapas posteriores.

## 6. Validación histórica de D contra benchmarks de P

Este es el paso central de la Etapa 2. Antes de muestrear, comparamos la distribución empírica del dataset D contra valores históricos verificables de la población P. La pregunta operativa es: ¿D refleja a P, o muestra sesgos estructurales que la muestra M debería corregir? Si D se desvía dentro de tolerancia, podemos extraer M con asignación calibrada por la probabilidad empírica `p_D`. Si se desvía fuera de tolerancia en una o más métricas, M debe corregir, no replicar el sesgo.

Métricas, fuentes y tolerancias acordadas:

| Métrica | Esperado en P | Fuente | Tolerancia |
|---|---|---|---|
| Manhattan + airport pickups | ~95% | TLC HAIL Market Analysis 2013 | ±3 pp |
| Airport pickups | ~8% del total | Row Zero analysis sobre `airport_fee` | ±2 pp |
| Outer boroughs pickups | ~5% | TLC HAIL 2013 | ±3 pp |
| Mean fare (metered: credit + cash) | USD 13.47 | TLC Flex Fare Pilot Evaluation 2023 | ±10% |
| Mean distance (metered) | 3.15 mi | TLC Flex Fare Pilot Evaluation 2023 | ±15% |
| Distance bins (short / medium / long) | 41.33% / 57.49% / 1.18% | Riascos y Mateos 2020 | ±5 pp por bin |
| Pearson r(distance, fare) sobre metered | > 0.7 | sanity check post-limpieza | floor 0.7 |
| Credit share entre metered | 75-78% | Donkor 2020 Stanford | ±5 pp |
| Flex Fare share global | crecimiento post sep 2024 | TLC Annual Report 2024 | confirmar tendencia |

Notas metodológicas:

- "Metered" significa pagos donde el taxímetro está activo: `payment_group in {credit, cash}`. El régimen Flex Fare no se incluye en métricas de tarifa porque el modelo upfront no usa taxímetro y sus tarifas se acuerdan antes del viaje.
- Las tolerancias son blandas: las fuentes son de períodos distintos (TLC HAIL es 2013, Riascos y Mateos usaron 2010-2013, Donkor analizó 2017-2019) y el dataset 2024-2025 incluye efectos post-COVID y la introducción del régimen Flex Fare en septiembre de 2024.
- Una desviación fuera de tolerancia no invalida D, indica que la muestra M debe calibrarse para corregir el sesgo o que el análisis posterior debe documentar la limitación.

### 6.1 Distribuciones empíricas en D

Calculamos todas las métricas en una sola pasada agregada (11 expresiones, dentro del límite seguro del heap por defecto). Una sola pasada sobre 84 millones de filas es más barata que tres `groupBy` separados.

In [17]:
n_feat = df_feat.count()

geo_vals = ["manhattan", "airport", "outer_borough", "unknown"]
pay_vals = ["credit", "cash", "flex", "other"]
dist_vals = ["short", "medium", "long"]

agg_exprs = []
for v in geo_vals:
    agg_exprs.append(F.sum((F.col("pu_macro_zone") == v).cast("int")).alias(f"geo_{v}"))
for v in pay_vals:
    agg_exprs.append(F.sum((F.col("payment_group") == v).cast("int")).alias(f"pay_{v}"))
for v in dist_vals:
    agg_exprs.append(F.sum((F.col("trip_distance_bin") == v).cast("int")).alias(f"dist_{v}"))

# Mean fare y mean distance sobre el subconjunto metered, conditional avg en la misma pasada.
agg_exprs.append(
    F.avg(F.when(F.col("payment_group").isin("credit", "cash"), F.col("fare_amount"))).alias("metered_mean_fare")
)
agg_exprs.append(
    F.avg(F.when(F.col("payment_group").isin("credit", "cash"), F.col("trip_distance"))).alias("metered_mean_distance")
)

counts = df_feat.agg(*agg_exprs).first()

geo_pct = {v: counts[f"geo_{v}"] / n_feat * 100 for v in geo_vals}
pay_pct = {v: counts[f"pay_{v}"] / n_feat * 100 for v in pay_vals}
dist_pct = {v: counts[f"dist_{v}"] / n_feat * 100 for v in dist_vals}

mean_fare = counts["metered_mean_fare"]
mean_dist = counts["metered_mean_distance"]
n_metered = counts["pay_credit"] + counts["pay_cash"]
credit_share_metered = counts["pay_credit"] / n_metered * 100

print(f"n_feat (post-limpieza e imputación): {n_feat:,}")
print(f"n_metered (credit + cash): {n_metered:,}\n")

print(f"Distribución geográfica:")
for v in geo_vals:
    print(f"  {v:<14}: {geo_pct[v]:>6.2f}%")

print(f"\nDistribución de pago:")
for v in pay_vals:
    print(f"  {v:<14}: {pay_pct[v]:>6.2f}%")

print(f"\nDistribución de distancia:")
for v in dist_vals:
    print(f"  {v:<14}: {dist_pct[v]:>6.2f}%")

print(f"\nMétricas sobre metered:")
print(f"  Mean fare       : USD {mean_fare:.2f}")
print(f"  Mean distance   : {mean_dist:.2f} mi")
print(f"  Credit share    : {credit_share_metered:.2f}%")

n_feat (post-limpieza e imputación): 84,437,138
n_metered (credit + cash): 70,639,521

Distribución geográfica:
  manhattan     :  87.50%
  airport       :   7.43%
  outer_borough :   4.83%
  unknown       :   0.25%

Distribución de pago:
  credit        :  72.22%
  cash          :  11.44%
  flex          :  14.86%
  other         :   1.48%

Distribución de distancia:
  short         :  26.81%
  medium        :  67.63%
  long          :   5.55%

Métricas sobre metered:
  Mean fare       : USD 19.66
  Mean distance   : 3.41 mi
  Credit share    : 86.33%


In [18]:
# Tabla de validación: observado vs esperado, con estado OK/FAIL por métrica.

def chequeo(observado, esperado, tolerancia, modo):
    """Devuelve OK si la desviación cabe en tolerancia, FAIL si no.

    modo='pp'   : tolerancia en puntos porcentuales (|obs - esp| <= tol)
    modo='pct'  : tolerancia en porcentaje relativo (|obs - esp| / esp * 100 <= tol)
    modo='floor': el observado debe superar el umbral esperado
    """
    if modo == "pp":
        return "OK" if abs(observado - esperado) <= tolerancia else "FAIL"
    if modo == "pct":
        return "OK" if abs(observado - esperado) / esperado * 100 <= tolerancia else "FAIL"
    if modo == "floor":
        return "OK" if observado > esperado else "FAIL"
    return "?"

metricas = [
    ("Manhattan + airport pct",     geo_pct["manhattan"] + geo_pct["airport"], 95.0,  3.0, "pp"),
    ("Airport pct",                 geo_pct["airport"],                          8.0,  2.0, "pp"),
    ("Outer borough pct",           geo_pct["outer_borough"],                    5.0,  3.0, "pp"),
    ("Mean fare metered (USD)",     mean_fare,                                  13.47, 10.0, "pct"),
    ("Mean distance metered (mi)",  mean_dist,                                   3.15, 15.0, "pct"),
    ("trip_distance short pct",     dist_pct["short"],                          41.33, 5.0, "pp"),
    ("trip_distance medium pct",    dist_pct["medium"],                         57.49, 5.0, "pp"),
    ("trip_distance long pct",      dist_pct["long"],                            1.18, 5.0, "pp"),
    ("Credit share metered (pct)",  credit_share_metered,                       76.5,  5.0, "pp"),
]

print(f"{'Métrica':<32} {'Observado':>12} {'Esperado':>12} {'Tol':>7} {'Estado':>8}")
print("-" * 75)
for nombre, obs, esp, tol, modo in metricas:
    st = chequeo(obs, esp, tol, modo)
    if modo == "pct":
        print(f"{nombre:<32} {obs:>12.2f} {esp:>12.2f} {tol:>6.1f}% {st:>8}")
    else:
        print(f"{nombre:<32} {obs:>12.2f} {esp:>12.2f} {tol:>6.1f}pp {st:>8}")

# Flex Fare share es informativo (sin esperado fijo, se valida como tendencia post sep 2024).
print(f"\nFlex Fare share global: {pay_pct['flex']:.2f}% (informativo; tendencia esperada: crecimiento post-2024-09)")

Métrica                             Observado     Esperado     Tol   Estado
---------------------------------------------------------------------------
Manhattan + airport pct                 94.92        95.00    3.0pp       OK
Airport pct                              7.43         8.00    2.0pp       OK
Outer borough pct                        4.83         5.00    3.0pp       OK
Mean fare metered (USD)                 19.66        13.47   10.0%     FAIL
Mean distance metered (mi)               3.41         3.15   15.0%       OK
trip_distance short pct                 26.81        41.33    5.0pp     FAIL
trip_distance medium pct                67.63        57.49    5.0pp     FAIL
trip_distance long pct                   5.55         1.18    5.0pp       OK
Credit share metered (pct)              86.33        76.50    5.0pp     FAIL

Flex Fare share global: 14.86% (informativo; tendencia esperada: crecimiento post-2024-09)


In [19]:
# Pearson r entre trip_distance y fare_amount sobre el subconjunto metered.
# Se calcula en una pasada separada porque F.corr no acepta argumentos condicionales.

metered = df_feat.filter(F.col("payment_group").isin("credit", "cash"))
pearson_r = metered.agg(F.corr("trip_distance", "fare_amount").alias("r")).first()["r"]

print(f"Pearson r(trip_distance, fare_amount) sobre metered: {pearson_r:.3f}")
print(f"Esperado: > 0.7 (sanity check post-limpieza)")
print(f"Estado: {'OK' if pearson_r > 0.7 else 'FAIL'}")

Pearson r(trip_distance, fare_amount) sobre metered: 0.930
Esperado: > 0.7 (sanity check post-limpieza)
Estado: OK


### 6.2 Lectura de la validación

La validación arroja 5 OK y 4 FAIL sobre 9 métricas comparables, más un Pearson r de 0.930 (muy por encima del piso de 0.7 acordado). Las desviaciones se concentran en métricas sensibles a inflación, regulación post-pandemia y cambio en hábitos de pago. Ninguna apunta a un sesgo del muestreo, a una falla de la limpieza ni a un problema del diseño del estrato; todas tienen atribución externa documentable.

**Bloque 1: distribución geográfica — OK en las tres métricas.** Manhattan + airport queda en 94.92% (vs 95% esperado, diferencia 0.08 pp), airport solo en 7.43% (vs aproximadamente 8%), outer borough en 4.83% (vs 5%). La concentración geográfica del Yellow Taxi en 2024-2025 es prácticamente idéntica a la documentada por TLC HAIL 2013, consistente con que la flota medallón sigue físicamente concentrada en Manhattan y los aeropuertos por la regulación TLC, que no ha cambiado en una década.

**Bloque 2: tarifas y correlación — FAIL en mean fare, OK en mean distance y Pearson r.** El mean fare metered observado es USD 19.66 contra USD 13.47 esperado (+46%, muy fuera de la tolerancia de ±10%). Causa atribuible: el benchmark de TLC Flex Fare Pilot Evaluation 2023 se construyó sobre datos del orden de 2018-2022. Desde entonces hubo tres cambios estructurales acumulativos:

- **Agosto 2022: alza tarifaria TLC.** La tarifa base subió de USD 2.50 a USD 3.00 y la tasa por milla de USD 2.50 a USD 3.50, un incremento del orden del 20% en el ticket promedio.
- **Enero 2025: cargo Central Business District.** Entró en vigor el CBD Congestion Fee de USD 2.50 por viaje en el Manhattan core, aplicable a aproximadamente 88% de los pickups del dataset (que son justamente Manhattan).
- **Inflación general 2022-2025** acumulada cercana al 12-15%, que se traslada a las tarifas planas de aeropuerto y a propinas.

Sumadas, las tres explican un alza del orden del 40-50% sobre el ticket promedio del benchmark, consistente con el +46% observado. **No es sesgo del muestreo, es drift regulatorio e inflacionario.** La mean distance de 3.41 mi contra 3.15 mi esperado queda dentro de ±15%: el viaje promedio sigue siendo el mismo en kilometraje, lo que cambió es el precio. El Pearson r de 0.930 confirma que la limpieza de outliers en sección 3 restauró la relación estructural distancia-tarifa que Etapa 1 había visto colapsada a 0.01 en datos crudos por la presencia de outliers extremos.

**Bloque 3: distribución de distancia y credit share — FAIL explicado por dos causas combinadas.** Los distance bins quedaron en 26.81% short / 67.63% medium / 5.55% long contra los 41.33% / 57.49% / 1.18% de Riascos y Mateos 2020. Dos causas combinadas:

- **Definición de distancia distinta:** Riascos y Mateos midieron distancia geográfica zona-a-zona (centroide-a-centroide), mientras que `trip_distance` en el dataset TLC viene del odómetro e incluye desvíos por tráfico, vueltas a la cuadra y recorridos por sentido único. Para viajes cortos la diferencia entre ambas medidas es proporcionalmente mayor (un viaje de 0.9 mi geográficas puede registrar 1.3 mi por odómetro), lo que migra una fracción importante del bin "short" al bin "medium". Esto solo explicaría un sesgo direccional, no la magnitud completa.
- **Displacement por servicios FHV (Uber, Lyft, Via):** Riascos y Mateos usaron datos 2010-2013, antes de la dominancia FHV en Manhattan. Los viajes cortos intra-Manhattan (los más sustituibles por app-based) migraron masivamente a Uber/Lyft entre 2014 y 2024, dejando al Yellow Taxi con una mezcla cargada hacia viajes a aeropuerto, intercity bound y trayectos medios. Esto reduce la fracción short y aumenta long en términos relativos.

El credit share metered observado es 86.33% contra el rango histórico de 75-78% de Donkor 2020 Stanford. La diferencia se atribuye al crecimiento secular del pago electrónico: Donkor analizó 2017-2019; entre 2020 y 2025 la pandemia aceleró el uso de tarjeta y tap-to-pay y la TLC mandó como obligatoria la capacidad de tap-to-pay post-2022. **No es sesgo, es tendencia social documentada.**

El Flex Fare share global de 14.86% es consistente con que el régimen entró en vigor en septiembre de 2024 y nuestro dataset cubre 24 meses (8 meses pre-Flex en 2024 + 4 meses post-Flex en 2024 + 12 meses de 2025). Si solo midiéramos septiembre 2024 en adelante esperaríamos un share cercano al 20-25%; el promedio diluido por los 8 meses sin Flex cae al 14.86%, dentro de la tendencia esperada.

### Decisión: proceder con muestreo calibrado por `p_D` empírica

Las cuatro métricas que salieron FAIL tienen explicación atribuible a cambios regulatorios documentados, inflación acumulada, diferencia de definición de medida y tendencia secular de pagos. Ninguna FAIL apunta a un problema del dataset, de la limpieza o del diseño del estrato. El Pearson r de 0.930 confirma integridad estructural de las relaciones tarifa-distancia tras la limpieza de Etapa 3.

La lectura operativa es que **D refleja a P_2024-2025**, no a P_2013-2019 (que es la población que miden las fuentes históricas). Para la sección 8 (cálculo del diccionario de fracciones de muestreo) usaremos `p_D` empírica del estrato sin calibración correctiva contra los benchmarks históricos. La asignación calibrada del muestreo solo aplicará el piso mínimo por celda para preservar estratos raros (airport + cash + late_night y similares), no para corregir desviaciones que ya no son sesgo sino realidad poblacional actual.

Las desviaciones quedan documentadas como hallazgos de Etapa 2: el dataset 2024-2025 difiere de las distribuciones publicadas para 2010-2019 por causas estructurales identificadas, lo cual es información sustantiva tanto para la evaluación del proyecto como para etapas posteriores. La sección 7 (cálculo de probabilidades empíricas por estrato) toma esta decisión como insumo y calcula `p_D` sobre `df_feat` tal cual.

## 7. Cálculo de probabilidades empíricas y enumeración del estrato

Cada celda del estrato compuesto `stratum_id` tiene una probabilidad de ocurrencia que necesitamos para el cálculo de fracciones de muestreo en la sección 8 (cálculo del diccionario de fracciones de muestreo). Presentamos las probabilidades de dos formas complementarias, alineadas con la convención de la rúbrica:

1. **Producto de marginales `p_product(s) = p(pu_macro_zone=a) × p(payment_group=b) × p(day_hour_bucket=c) × p(trip_distance_bin=d)`**, que sería la probabilidad teórica de la celda **si las cuatro variables fueran independientes**. Este es el esquema que ilustra el ejemplo de la rúbrica (donde `p(A=a, B=b) = p(A=a) × p(B=b) = 0.3 × 0.2 = 0.06`).

2. **Probabilidad empírica conjunta `p_D(s) = n_D(s) / n_total`**, donde `n_D(s)` es el conteo observado de la combinación `s` en `df_feat`. Esta es la distribución real, sin asumir independencia.

**Por qué presentamos ambas y por qué usamos la empírica para el muestreo:** las cuatro variables del estrato **no son independientes**, y los datos lo confirmarán abajo. Por ejemplo:

- `pu_macro_zone = airport` está correlacionado con `trip_distance_bin = long` (viajes desde JFK, LGA y EWR a Manhattan son por construcción de varios kilómetros).
- `payment_group = flex` no coexiste con `pu_macro_zone = airport` (Port Authority prohíbe e-hail en los aeropuertos según TLC Flex Fare Pilot Evaluation 2023), produciendo combinaciones estructuralmente imposibles.

Si usáramos `p_product` para la asignación de fracciones, sobre-asignaríamos probabilidad a celdas imposibles (como `flex × airport`) y subestimaríamos celdas correlacionadas (como `airport × long`). La asignación de fracciones de la sección 8 usa por lo tanto `p_D` empírica. La presentación de `p_product` queda como referencia metodológica (cumple la convención de la rúbrica) y como diagnóstico de dependencia entre variables.

Enumeramos explícitamente las 4 × 4 × 5 × 3 = 240 combinaciones teóricas y reportamos para cada una: `n_D`, `p_D`, `p_product` y el ratio `p_D / p_product` (que mide cuán lejos del modelo de independencia está la celda observada). Las combinaciones con `n_D = 0` se reportan como vacías con probabilidad cero, no se omiten.

In [20]:
import itertools

cols_vals = [
    ("pu_macro_zone",     ["airport", "manhattan", "outer_borough", "unknown"]),
    ("payment_group",     ["credit", "cash", "flex", "other"]),
    ("day_hour_bucket",   ["late_night", "weekend", "weekday_am", "weekday_pm_peak", "other"]),
    ("trip_distance_bin", ["short", "medium", "long"]),
]

# Una sola pasada agregada: 16 conditional sums + total = 17 expresiones, dentro del límite del heap.
agg_exprs = [F.count("*").alias("__total__")]
for col, vals in cols_vals:
    for v in vals:
        agg_exprs.append(F.sum((F.col(col) == v).cast("int")).alias(f"{col}__{v}"))

agg_row = df_feat.agg(*agg_exprs).first()
total = agg_row["__total__"]

marginales = {}
for col, vals in cols_vals:
    marginales[col] = {v: agg_row[f"{col}__{v}"] / total for v in vals}

print(f"Total de filas en df_feat: {total:,}\n")
print("Marginales por variable de estratificación:\n")
for col, vals in cols_vals:
    print(f"{col}:")
    for v in vals:
        prob = marginales[col][v]
        print(f"  p({col} = {v!r:<20}) = {prob:.6f} ({prob*100:6.2f}%)")
    print()

Total de filas en df_feat: 84,437,138

Marginales por variable de estratificación:

pu_macro_zone:
  p(pu_macro_zone = 'airport'           ) = 0.074263 (  7.43%)
  p(pu_macro_zone = 'manhattan'         ) = 0.874987 ( 87.50%)
  p(pu_macro_zone = 'outer_borough'     ) = 0.048288 (  4.83%)
  p(pu_macro_zone = 'unknown'           ) = 0.002463 (  0.25%)

payment_group:
  p(payment_group = 'credit'            ) = 0.722234 ( 72.22%)
  p(payment_group = 'cash'              ) = 0.114359 ( 11.44%)
  p(payment_group = 'flex'              ) = 0.148591 ( 14.86%)
  p(payment_group = 'other'             ) = 0.014816 (  1.48%)

day_hour_bucket:
  p(day_hour_bucket = 'late_night'        ) = 0.084615 (  8.46%)
  p(day_hour_bucket = 'weekend'           ) = 0.235082 ( 23.51%)
  p(day_hour_bucket = 'weekday_am'        ) = 0.136216 ( 13.62%)
  p(day_hour_bucket = 'weekday_pm_peak'   ) = 0.231463 ( 23.15%)
  p(day_hour_bucket = 'other'             ) = 0.312623 ( 31.26%)

trip_distance_bin:
  p(trip_distance_

In [21]:
n_feat = total

D_dist = (df_feat.groupBy("stratum_id").count()
          .withColumnRenamed("count", "n_D")
          .withColumn("p_D", F.col("n_D") / F.lit(n_feat)))

n_combinaciones_no_vacias = D_dist.count()
print(f"Combinaciones del estrato observadas en df_feat: {n_combinaciones_no_vacias} de 240 teóricas")
print(f"Combinaciones vacías:                            {240 - n_combinaciones_no_vacias}\n")

print("Top 10 estratos por probabilidad empírica (dominantes):")
D_dist.orderBy(F.desc("p_D")).show(10, truncate=False)

print("Bottom 10 estratos no vacíos por probabilidad empírica (raros poblados):")
D_dist.orderBy("p_D").show(10, truncate=False)

Combinaciones del estrato observadas en df_feat: 240 de 240 teóricas
Combinaciones vacías:                            0

Top 10 estratos por probabilidad empírica (dominantes):


+---------------------------------------+--------+--------------------+
|stratum_id                             |n_D     |p_D                 |
+---------------------------------------+--------+--------------------+
|manhattan|credit|other|medium          |11515883|0.13638409913893576 |
|manhattan|credit|weekday_pm_peak|medium|9088533 |0.10763667759558596 |
|manhattan|credit|weekend|medium        |8179141 |0.09686662994191016 |
|manhattan|credit|other|short           |5434482 |0.06436127666951479 |
|manhattan|credit|weekday_am|medium     |4743580 |0.05617883448394473 |
|manhattan|credit|weekday_pm_peak|short |4531816 |0.053670885908046764|
|manhattan|credit|weekend|short         |3698314 |0.043799613388127864|
|manhattan|credit|late_night|medium     |2866367 |0.03394675693531915 |
|manhattan|credit|weekday_am|short      |2431762 |0.028799673432796834|
|manhattan|flex|other|medium            |2386166 |0.02825967407848428 |
+---------------------------------------+--------+--------------

+----------------------------------+---+---------------------+
|stratum_id                        |n_D|p_D                  |
+----------------------------------+---+---------------------+
|unknown|flex|late_night|short     |53 |6.276858886429808E-7 |
|airport|flex|weekday_pm_peak|short|54 |6.395290186173766E-7 |
|airport|flex|other|short          |59 |6.987446684893559E-7 |
|airport|flex|weekday_am|short     |60 |7.105877984637518E-7 |
|unknown|other|weekday_am|long     |75 |8.882347480796898E-7 |
|airport|flex|late_night|short     |79 |9.356072679772732E-7 |
|airport|flex|weekend|short        |80 |9.474503979516691E-7 |
|unknown|other|late_night|long     |90 |1.0658816976956277E-6|
|unknown|other|weekday_pm_peak|long|103|1.219842387362774E-6 |
|unknown|other|weekend|long        |107|1.2672149072603575E-6|
+----------------------------------+---+---------------------+
only showing top 10 rows


In [22]:
# Colectamos D_dist al driver (máximo 240 filas, operación liviana) y enumeramos
# las 240 combinaciones teóricas en producto cartesiano. Las vacías (n_D = 0) se
# reportan explícitamente como pide la rúbrica ("todas las combinaciones posibles").

empirical_lookup = {
    r["stratum_id"]: (r["n_D"], float(r["p_D"]))
    for r in D_dist.collect()
}

rows = []
for g, p, d, dist in itertools.product(
    [v for _, vals in cols_vals for v in vals if _ == "pu_macro_zone"] or ["airport", "manhattan", "outer_borough", "unknown"],
    ["credit", "cash", "flex", "other"],
    ["late_night", "weekend", "weekday_am", "weekday_pm_peak", "other"],
    ["short", "medium", "long"],
):
    sid = f"{g}|{p}|{d}|{dist}"
    p_product = (marginales["pu_macro_zone"][g]
                 * marginales["payment_group"][p]
                 * marginales["day_hour_bucket"][d]
                 * marginales["trip_distance_bin"][dist])
    n_D, p_D = empirical_lookup.get(sid, (0, 0.0))
    ratio = (p_D / p_product) if p_product > 0 else None
    rows.append({
        "stratum_id": sid,
        "pu": g, "pay": p, "day": d, "dist": dist,
        "n_D": n_D,
        "p_D": p_D,
        "p_product": p_product,
        "ratio_D_over_product": ratio,
        "estado": "vacío" if n_D == 0 else "poblado",
    })

import pandas as pd
strata = pd.DataFrame(rows)
print(f"Combinaciones totales enumeradas: {len(strata)} (esperado 240)")
print(f"Pobladas:                          {(strata['estado'] == 'poblado').sum()}")
print(f"Vacías:                            {(strata['estado'] == 'vacío').sum()}\n")

# Patrón de combinaciones vacías agrupadas por (pu_macro_zone, payment_group):
# si aparecen 15 celdas vacías para un (pu, pay), significa que TODA esa combinación es imposible.
import collections
patron_vacias = collections.Counter()
for _, r in strata[strata["estado"] == "vacío"].iterrows():
    patron_vacias[(r["pu"], r["pay"])] += 1

print("Patrones de combinaciones vacías (pu_macro_zone × payment_group):")
print(f"{'pu_macro_zone':<16} {'payment':<8} {'vacías':>8} {'/ 15 max':>10}")
print("-" * 50)
for (pu, pay), n in sorted(patron_vacias.items(), key=lambda x: -x[1]):
    print(f"{pu:<16} {pay:<8} {n:>8} {' (total)' if n == 15 else ''}")

# Dependencia entre variables: el ratio p_D / p_product mide cuánto se separa la celda
# observada del modelo de independencia.
print("\nTop 5 estratos con mayor dependencia POSITIVA (ratio p_D / p_product > 1):")
top_pos = strata[strata["estado"] == "poblado"].nlargest(5, "ratio_D_over_product")
print(top_pos[["stratum_id", "n_D", "p_D", "p_product", "ratio_D_over_product"]].to_string(index=False))

print("\nTop 5 estratos con mayor dependencia NEGATIVA (ratio p_D / p_product < 1):")
bot_neg = strata[(strata["estado"] == "poblado") & (strata["ratio_D_over_product"] > 0)].nsmallest(5, "ratio_D_over_product")
print(bot_neg[["stratum_id", "n_D", "p_D", "p_product", "ratio_D_over_product"]].to_string(index=False))

Combinaciones totales enumeradas: 240 (esperado 240)
Pobladas:                          240
Vacías:                            0

Patrones de combinaciones vacías (pu_macro_zone × payment_group):
pu_macro_zone    payment    vacías   / 15 max
--------------------------------------------------

Top 5 estratos con mayor dependencia POSITIVA (ratio p_D / p_product > 1):
                        stratum_id   n_D      p_D  p_product  ratio_D_over_product
airport|other|weekday_pm_peak|long 21801 0.000258   0.000014             18.250722
        airport|other|weekend|long 20360 0.000241   0.000014             16.782043
     airport|other|late_night|long  7172 0.000085   0.000005             16.423913
          airport|other|other|long 25469 0.000302   0.000019             15.786150
outer_borough|flex|late_night|long 41345 0.000490   0.000034             14.518748

Top 5 estratos con mayor dependencia NEGATIVA (ratio p_D / p_product < 1):
                        stratum_id  n_D          p_D  p_p

### 7.1 Lectura del enumerado

La salida cuantifica tres hallazgos relevantes para la sección 8 (cálculo del diccionario de fracciones de muestreo):

**1. Cardinalidad observada: 240 de 240 combinaciones pobladas (100% de ocupación).** El plan esperaba entre 180 y 220 pobladas, asumiendo que algunas combinaciones (como `flex × airport`) serían estructuralmente imposibles y aparecerían como vacías. Los datos 2024-2025 muestran que **ninguna combinación está completamente vacía**: las 240 celdas teóricas tienen al menos algunos registros. Operativamente, el diccionario de fracciones de la sección 8 contendrá 240 entradas, no menos.

**2. Las combinaciones que se esperaban imposibles son cuasi-vacías, no vacías.** La hipótesis a priori era que `flex × airport × * × *` (15 celdas) estuviera vacía porque Port Authority prohíbe el régimen e-hail en los aeropuertos según TLC Flex Fare Pilot Evaluation 2023. Los datos matizan la hipótesis: esas celdas existen pero son **estructuralmente raras** (entre 53 y 80 registros cada una, ratios `p_D / p_product` entre 0.00076 y 0.00374, dos a tres órdenes de magnitud por debajo del modelo de independencia). Los cinco estratos con mayor dependencia negativa son **todos** del patrón `airport × flex × * × short`:

| stratum_id | n_D | ratio p_D / p_product |
|---|---|---|
| `airport \| flex \| other \| short` | 59 | 0.00076 |
| `airport \| flex \| weekday_pm_peak \| short` | 54 | 0.00093 |
| `airport \| flex \| weekend \| short` | 80 | 0.00136 |
| `airport \| flex \| weekday_am \| short` | 60 | 0.00176 |
| `airport \| flex \| late_night \| short` | 79 | 0.00374 |

**Interpretación:** la prohibición Port Authority no es absoluta sino una restricción muy fuerte; algunos registros residuales (probablemente excepciones reglamentarias o errores de codificación de zona) sobreviven. Para el muestreo estos estratos quedan protegidos por el piso mínimo de la sección 8: al ser tan pequeños se conservarán completos.

**3. Dependencias positivas dominantes confirman la utilidad de usar `p_D` empírica en vez de `p_product`.** Los cinco estratos con mayor dependencia positiva son:

| stratum_id | n_D | ratio p_D / p_product |
|---|---|---|
| `airport \| other \| weekday_pm_peak \| long` | 21,801 | 18.25 |
| `airport \| other \| weekend \| long` | 20,360 | 16.78 |
| `airport \| other \| late_night \| long` | 7,172 | 16.42 |
| `airport \| other \| other \| long` | 25,469 | 15.79 |
| `outer_borough \| flex \| late_night \| long` | 41,345 | 14.52 |

Cuatro de las cinco celdas son `airport × other × * × long`, con ratios entre 15.79 y 18.25. Interpretación: los viajes desde aeropuertos a Manhattan son largos por construcción (JFK, LGA, EWR están a 10 a 20 millas del Manhattan core), y el patrón se concentra en `payment_group = "other"`, lo cual sugiere fuertemente que la tarifa plana JFK (`RatecodeID = 2`, ahora USD 70 + recargos) se codifica con un `payment_type` distinto al crédito estándar (probablemente alguno de los códigos 3 a 6 según el reporte del TPEP).

La quinta celda, `outer_borough × flex × late_night × long` con 41,345 registros, representa viajes app-based en madrugada desde Brooklyn, Queens, Bronx o Staten Island hacia destinos lejanos (típicamente al aeropuerto o a Manhattan). Es el uso característico de Flex Fare como reemplazo de FHV: pickups en outer boroughs en horarios donde el yellow taxi tradicional escasea.

Si hubiéramos usado `p_product` para asignar fracciones de muestreo, estos cinco estratos quedarían sub-muestreados por factor 14 a 18, perdiendo señal valiosa sobre los perfiles airport, outer borough nocturno y long-distance. **La empírica `p_D` los preserva en su proporción real.**

### Concentración del estrato y consecuencias para el muestreo

Los 10 estratos dominantes son todos `manhattan × credit × * × *` y concentran aproximadamente 65% del dataset. El más dominante es `manhattan | credit | other | medium` con 13.64% (aproximadamente 11.5 millones de filas), seguido por `manhattan | credit | weekday_pm_peak | medium` con 10.76%. Esto es consistente con el perfil característico Yellow Taxi: pickup en Manhattan, pago con tarjeta, horario laboral o pico vespertino, distancia media.

Los estratos más raros poblados tienen del orden de 50 a 100 registros cada uno. Con `MIN_PER_STRATUM = 500` que se aplica en la sección 8, estos estratos se tomarán **completos** (`fraction = 1.0`), garantizando que sobrevivan al muestreo. Sin piso mínimo, un muestreo proporcional puro asignaría a una celda de 53 registros una muestra esperada de 53 × (5,000,000 / 84,437,138) ≈ 3 filas, perdiendo efectivamente el perfil. El piso mínimo es lo que distingue al esquema calibrado de uno proporcional puro.

La sección 8 (cálculo del diccionario de fracciones de muestreo) recibe `D_dist` (Spark DataFrame con 240 filas: `stratum_id`, `n_D`, `p_D`) y aplica la fórmula calibrada `target_n(s) = min(n_D(s), max(MIN_PER_STRATUM, round(N_M × p_D(s))))` para producir el diccionario de 240 fracciones que alimenta `sampleBy` en la sección 9 (extracción de la muestra M).